# M19/C3 model-comparison reproduction

This notebook is a separate, code-preserving reproduction supplement for the
M19/C3 internal model-comparison audit. It does **not** replace, amend, or
re-run the certified frozen validation release.

## Scope

- Stage 6T45 is an internal, galaxy-separated out-of-fold comparison on the
  already analysed SPARC data. It is post-selection explanatory work, not an
  external prospective validation or a fundamental field equation.
- M19 is the code label `M19_full_free_q_saturated_gradient`.
- C3 is the code label `C3_joint_q05_linear_gradient`: fixed $q=0.5$, linear
  (unsaturated) response, with the nonlocal source and radial-gradient term.

## Exact model definition implemented by Stage 6T45

\begin{align}
g_{\rm cum}(r) &= \frac{2}{r^2}\int_0^r r' g_{\rm bar}(r')\,dr',\\
\omega(r) &= g_{\rm bar}(r)^{1-\eta}g_{\rm cum}(r)^{\eta},\\
s_q(r) &= g_{\rm ref}\left(\frac{\omega(r)}{g_{\rm ref}}\right)^q,\\
\Delta g_{\rm M19}(r) &= A g_{\max}\tanh\left(\frac{s_q(r)}{g_{\max}}\right),\\
g_{\rm pred}^{\rm M19}(r) &= g_{\rm bar}(r)+\Delta g_{\rm M19}(r)
+\lambda\frac{d}{dr}[r\Delta g_{\rm M19}(r)].
\end{align}

M19 has six fitted parameters: $A$, $g_{\rm ref}$, $g_{\max}$, $\eta$, $q$,
and $\lambda$. In the C3 specialization, saturation is removed and $q=0.5$
is fixed. With $K=A\sqrt{g_{\rm ref}}$, its response becomes
$\Delta g_{\rm C3}=K\sqrt{\omega}$. This is an exact reparameterization of
the linear C3 form, not an equivalence between C3 and the full free-$q$,
saturated M19 reference model.

Run the Stage 6T45 cell first; it creates the audited M19/C3 comparison.
Run the Stage 6T47 cell afterwards; it verifies the exact C3 algebraic mapping.


In [2]:
"""STUFE 6T45 -- M19-GESAMTTEST EINER GEMEINSAMEN MINIMALVARIANTE.

Interne Reproduzierbarkeits- und Zerlegungsprüfung auf bereits analysierten
SPARC-Daten. Dies ist kein neuer externer Test und keine fundamentale
UQSH-Feldgleichung.

Die radiale SPARC-Referenz enthält g_bar, aber keine galaxienweise variierende
und unabhängig auditierte Stern-M/L. Die publizierte feste Disk-/Bulge-M/L ist
bereits in g_bar enthalten. Deshalb testet diese Revision NICHT die
Populations-/M/L-Komponente: sie wäre mit einer konstanten oder erfundenen
Kovariate nicht identifizierbar. Der Test vergleicht M19 direkt mit der
gemeinsam vereinfachten Variante q=0.5 ohne Saettigung, bei beibehaltenem
nichtlokalem Anteil und radialem Gradient. Dies verändert keine Quelle.

WICHTIG: Die Variante wurde aus den Ergebnissen von 6T44B motiviert. Daher ist
dies ein explikativer, post-selektionaler Gesamtvergleich und kein vorher
registrierter, bestaetigender Test. Der p-Wert beantwortet nur die Frage, ob
die beiden bereits festgelegten Varianten in denselben OOF-Folds verschieden
abschneiden; er belegt keine globale Modellsuche.
"""
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import re
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from scipy.stats import wilcoxon
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore", category=RuntimeWarning)

try:
    from google.colab import drive  # type: ignore
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass


# -----------------------------------------------------------------------------
# 1. Fixed configuration. Change only INPUT_MASTER_OVERRIDE if automatic source
#    discovery stops because several different candidate tables are present.
# -----------------------------------------------------------------------------
STAGE = "6T45"
CODE_VERSION = "6T45.1-joint-minimal-model-20260730"
PROJECT = Path("/content/drive/MyDrive/UQSH_DarkMatter_Project")
AUDIT = PROJECT / "journal_audit"
# Canonical audited radial source. It contains the four directly observed or
# reconstructed radial quantities required below. Do not replace it with an
# OOF-prediction output or with a table created by this stage.
INPUT_MASTER_OVERRIDE: Path | None = PROJECT / "data_processed" / "sparc_radial_observables.csv"
OUTER_FOLDS = 5
SEED = 20260730
N_STARTS = 3
MAX_NFEV = 1800
N_BOOTSTRAP = 10000

REQUIRED_LOGICAL_COLUMNS = ("galaxy", "radius", "gbar", "gobs")
ALIASES = {
    "galaxy": ("galaxy", "galaxy_name", "name", "galaxy_key", "galaxy_key_norm"),
    "radius": ("r_kpc", "radius_kpc", "radius", "r"),
    "gbar": ("gbar_m_s2", "g_bar_m_s2", "gbar", "local__gbar_m_s2"),
    "gobs": ("gobs_m_s2", "g_obs_m_s2", "gobs", "local__gobs_m_s2"),
}

# A population weight is deliberately absent; the source has no independently
# varying M/L. The last model is the requested JOINT change: it combines the
# fixed q=0.5 choice with the removal of saturation. No results are added or
# summed; it is fitted and evaluated as one complete model on every fold.
MODELS = {
    "M19_full_free_q_saturated_gradient": {"nonlocal": True, "q": "free", "saturation": True, "gradient": True},
    "C1_fixed_q05_saturated_gradient": {"nonlocal": True, "q": 0.5, "saturation": True, "gradient": True},
    "C2_free_q_linear_gradient": {"nonlocal": True, "q": "free", "saturation": False, "gradient": True},
    "C3_joint_q05_linear_gradient": {"nonlocal": True, "q": 0.5, "saturation": False, "gradient": True},
}

REFERENCE_MODEL = "M19_full_free_q_saturated_gradient"
JOINT_MODEL = "C3_joint_q05_linear_gradient"


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def normal(text: object) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(text).lower())


def resolve_column(frame: pd.DataFrame, logical: str) -> str | None:
    normalized = {normal(column): str(column) for column in frame.columns}
    for candidate in ALIASES[logical]:
        if normal(candidate) in normalized:
            return normalized[normal(candidate)]
    return None


def candidate_score(path: Path) -> tuple[int, dict[str, str]]:
    try:
        head = pd.read_csv(path, nrows=6)
    except Exception:
        return -1, {}
    mapping = {key: resolve_column(head, key) for key in REQUIRED_LOGICAL_COLUMNS}
    return sum(value is not None for value in mapping.values()), {
        key: value for key, value in mapping.items() if value is not None
    }


def locate_source() -> tuple[Path, dict[str, str], list[dict[str, object]]]:
    if INPUT_MASTER_OVERRIDE is not None:
        if not INPUT_MASTER_OVERRIDE.is_file():
            raise FileNotFoundError(f"INPUT_MASTER_OVERRIDE does not exist: {INPUT_MASTER_OVERRIDE}")
        frame = pd.read_csv(INPUT_MASTER_OVERRIDE, nrows=6)
        mapping = {key: resolve_column(frame, key) for key in REQUIRED_LOGICAL_COLUMNS}
        missing = [key for key, value in mapping.items() if value is None]
        if missing:
            raise ValueError(f"Override misses required logical columns: {missing}; columns: {list(frame.columns)}")
        return INPUT_MASTER_OVERRIDE, mapping, []

    # Only source-like master/radial tables are considered. Prior output tables
    # from this exact stage family are excluded to avoid self-ingestion.
    candidates: list[dict[str, object]] = []
    for path in sorted(PROJECT.rglob("*.csv")):
        low = str(path).lower()
        if "uqsh_stage6t44" in low or "stage6t44" in low:
            continue
        if not any(token in path.name.lower() for token in ("master", "radial", "sparc", "field")):
            continue
        score, mapping = candidate_score(path)
        if score >= 3:
            candidates.append({"path": str(path), "score": score, "mapping": json.dumps(mapping, sort_keys=True)})

    complete = [row for row in candidates if row["score"] == len(REQUIRED_LOGICAL_COLUMNS)]
    if len(complete) != 1:
        detail = "\n".join(json.dumps(row) for row in candidates[:50]) or "no usable CSV candidates"
        raise RuntimeError(
            "No unique radial master table with galaxy, radius, gbar and gobs was found. "
            "Set INPUT_MASTER_OVERRIDE to the already audited radial source. Candidates:\n" + detail
        )
    source = Path(str(complete[0]["path"]))
    head = pd.read_csv(source, nrows=6)
    mapping = {key: resolve_column(head, key) for key in REQUIRED_LOGICAL_COLUMNS}
    return source, mapping, candidates


def cumulative_gbar(radius: np.ndarray, gbar: np.ndarray) -> np.ndarray:
    """g_cum(r)=2/r^2 integral_0^r gbar(r') r' dr', per galaxy."""
    integral = np.zeros_like(radius, dtype=float)
    if len(radius) > 1:
        integrand = gbar * radius
        integral[1:] = np.cumsum(0.5 * (integrand[1:] + integrand[:-1]) * np.diff(radius))
    return np.maximum(2.0 * integral / np.maximum(radius**2, 1e-30), 1e-30)


def radial_gradient(frame: pd.DataFrame, response: np.ndarray) -> np.ndarray:
    """d[r*response]/dr, calculated independently within every galaxy."""
    out = np.full(len(frame), np.nan, dtype=float)
    for _, positions in frame.groupby("galaxy", sort=False).indices.items():
        positions = np.asarray(positions, dtype=int)
        radius = frame.radius.to_numpy(float)[positions]
        out[positions] = np.gradient(radius * response[positions], radius)
    return out


def prepare_source(source: Path, mapping: dict[str, str]) -> pd.DataFrame:
    raw = pd.read_csv(source)
    work = pd.DataFrame({key: pd.to_numeric(raw[column], errors="coerce") if key != "galaxy" else raw[column].astype(str)
                         for key, column in mapping.items()})
    work["galaxy"] = work["galaxy"].str.strip()
    work = work.replace([np.inf, -np.inf], np.nan).dropna()
    work = work[(work.radius > 0) & (work.gbar > 0) & (work.gobs > 0)].copy()
    work = work.sort_values(["galaxy", "radius"]).reset_index(drop=True)
    if work.galaxy.nunique() < OUTER_FOLDS * 4:
        raise RuntimeError("Too few galaxies after source cleaning for a five-fold galaxy-separated test.")
    if (work.groupby("galaxy").radius.diff().fillna(1) <= 0).any():
        raise RuntimeError("A galaxy has non-increasing radii; derivatives would be ambiguous.")
    work["gcum"] = work.groupby("galaxy", group_keys=False).apply(
        lambda x: pd.Series(cumulative_gbar(x.radius.to_numpy(), x.gbar.to_numpy()), index=x.index),
        include_groups=False,
    ).sort_index()
    work["log_gobs"] = np.log10(work.gobs)
    return work.reset_index(drop=True)


def parameter_spec(config: dict[str, object]) -> tuple[list[str], np.ndarray, np.ndarray, np.ndarray]:
    names = ["log10_A", "log10_gref", "log10_gmax"]
    x0 = [0.0, -10.5, -10.0]
    low = [-4.0, -14.0, -14.0]
    high = [4.0, -7.0, -7.0]
    if config["nonlocal"]:
        names.append("eta") ; x0.append(0.5); low.append(0.0); high.append(1.0)
    if config["q"] == "free":
        names.append("q"); x0.append(0.5); low.append(0.15); high.append(1.50)
    if config["gradient"]:
        names.append("lambda"); x0.append(0.0); low.append(-4.0); high.append(4.0)
    return names, np.asarray(x0), np.asarray(low), np.asarray(high)


def unpack(values: np.ndarray, names: list[str], config: dict[str, object]) -> dict[str, float]:
    p = dict(zip(names, values, strict=True))
    p.setdefault("eta", 0.0)
    p["q"] = float(config["q"]) if config["q"] != "free" else float(p["q"])
    p.setdefault("lambda", 0.0)
    p["A"] = 10.0 ** p.pop("log10_A")
    p["gref"] = 10.0 ** p.pop("log10_gref")
    p["gmax"] = 10.0 ** p.pop("log10_gmax")
    return p


def predict(frame: pd.DataFrame, values: np.ndarray, names: list[str], config: dict[str, object]) -> np.ndarray:
    p = unpack(values, names, config)
    log_omega = (1.0 - p["eta"]) * np.log(frame.gbar.to_numpy(float)) + p["eta"] * np.log(frame.gcum.to_numpy(float))
    omega = np.exp(np.clip(log_omega, -700, 700))
    omega_q = p["gref"] * np.power(np.maximum(omega / p["gref"], 1e-300), p["q"])
    if bool(config["saturation"]):
        delta = p["A"] * p["gmax"] * np.tanh(omega_q / p["gmax"])
    else:
        delta = p["A"] * omega_q
    if bool(config["gradient"]):
        grad = radial_gradient(frame, delta)
        delta = delta + p["lambda"] * grad
    return frame.gbar.to_numpy(float) + delta


def fit_model(train: pd.DataFrame, config: dict[str, object], seed: int) -> tuple[np.ndarray, list[str], dict[str, object]]:
    names, x0, low, high = parameter_spec(config)
    equal_galaxy_weight = 1.0 / train.groupby("galaxy").galaxy.transform("size").to_numpy(float)

    def residual(values: np.ndarray) -> np.ndarray:
        pred = predict(train, values, names, config)
        if np.any(~np.isfinite(pred)) or np.any(pred <= 0):
            return np.full(len(train), 1e6)
        return (np.log10(pred) - train.log_gobs.to_numpy(float)) * np.sqrt(equal_galaxy_weight)

    rng = np.random.default_rng(seed)
    starts = [x0]
    for _ in range(N_STARTS - 1):
        starts.append(low + rng.random(len(low)) * (high - low))
    best = None
    for start in starts:
        result = least_squares(residual, start, bounds=(low, high), max_nfev=MAX_NFEV, method="trf")
        score = float(np.mean(residual(result.x) ** 2))
        if best is None or score < best[0]:
            best = (score, result)
    assert best is not None
    result = best[1]
    return result.x, names, {
        "optimizer_success": bool(result.success), "optimizer_status": int(result.status),
        "optimizer_message": str(result.message), "nfev": int(result.nfev),
        "cost": float(result.cost), "lower_bounds": low, "upper_bounds": high,
    }


def metrics(y: np.ndarray, p: np.ndarray) -> dict[str, float]:
    """Metrics only for a completely finite, positive prediction subset."""
    if len(y) == 0:
        return {"rmse_dex": np.nan, "mae_dex": np.nan, "median_abs_error_dex": np.nan,
                "r2": np.nan, "bias_dex": np.nan}
    e = p - y
    return {
        "rmse_dex": float(np.sqrt(mean_squared_error(y, p))),
        "mae_dex": float(np.mean(np.abs(e))),
        "median_abs_error_dex": float(np.median(np.abs(e))),
        "r2": float(r2_score(y, p)), "bias_dex": float(np.mean(e)),
    }


def run_preflight() -> None:
    """Check the exact held-out-domain failure path before any Drive output."""
    probe = np.array([2.0e-11, 0.0, -3.0e-12, np.nan])
    valid = np.isfinite(probe) & (probe > 0)
    logged = np.full(probe.shape, np.nan, dtype=float)
    logged[valid] = np.log10(probe[valid])
    if valid.tolist() != [True, False, False, False] or not np.isfinite(logged[0]) or np.isfinite(logged[1:]).any():
        raise RuntimeError("Preflight failed: invalid OOF predictions are not safely recorded.")


def main() -> None:
    run_preflight()
    source, mapping, source_candidates = locate_source()
    data = prepare_source(source, mapping)
    run_id = "run_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
    root = AUDIT / "uqsh_stage6t45_m19_joint_minimal_model_groupcv" / run_id
    tables, figures, reports = root / "tables", root / "figures", root / "reports"
    for directory in (tables, figures, reports):
        directory.mkdir(parents=True, exist_ok=False)

    unique_galaxies = np.array(sorted(data.galaxy.unique()))
    splitter = GroupKFold(n_splits=OUTER_FOLDS)
    fold_for_galaxy: dict[str, int] = {}
    for fold, (_, test_idx) in enumerate(splitter.split(unique_galaxies, groups=unique_galaxies)):
        for galaxy in unique_galaxies[test_idx]:
            fold_for_galaxy[str(galaxy)] = fold
    data["outer_fold"] = data.galaxy.map(fold_for_galaxy).astype(int)

    all_predictions: list[pd.DataFrame] = []
    parameter_rows: list[dict[str, object]] = []
    bound_rows: list[dict[str, object]] = []
    for fold in range(OUTER_FOLDS):
        # Resetting the row index is essential: the radial-gradient calculation
        # indexes a fold-local response vector and must never inherit source-row
        # indices from the complete table.
        train = data[data.outer_fold != fold].copy().reset_index(drop=True)
        test = data[data.outer_fold == fold].copy().reset_index(drop=True)
        for model_index, (model_name, config) in enumerate(MODELS.items()):
            values, names, info = fit_model(train, config, SEED + 1000 * fold + model_index)
            prediction = predict(test, values, names, config)
            # A held-out gradient can make an additive response non-positive even
            # when the training fold was valid. This is a model-domain failure,
            # not a reason to discard the entire audit. Preserve it explicitly;
            # no clipping or replacement is allowed because that would invent a
            # different model and conceal its failure.
            prediction_valid = np.isfinite(prediction) & (prediction > 0)
            part = test[["galaxy", "radius", "gbar", "gobs", "gcum", "outer_fold"]].copy()
            part["model"] = model_name
            part["log_gobs"] = np.log10(part.gobs)
            part["gpred_raw"] = prediction
            part["prediction_valid"] = prediction_valid
            part["gpred"] = np.where(prediction_valid, prediction, np.nan)
            # Do not use np.where(np.log10(...)): NumPy evaluates both branches
            # and would still take log10 of invalid raw values.
            log_prediction = np.full(prediction.shape, np.nan, dtype=float)
            log_prediction[prediction_valid] = np.log10(prediction[prediction_valid])
            part["log_gpred"] = log_prediction
            part["residual_dex"] = part.log_gpred - part.log_gobs
            all_predictions.append(part)
            lower, upper = np.asarray(info["lower_bounds"]), np.asarray(info["upper_bounds"])
            for name, value, lo, hi in zip(names, values, lower, upper, strict=True):
                parameter_rows.append({"model": model_name, "fold": fold, "parameter": name, "value": float(value),
                                       "n_train_galaxies": int(train.galaxy.nunique()), "n_test_galaxies": int(test.galaxy.nunique()),
                                       "n_train_rows": len(train), "n_test_rows": len(test), "optimizer_success": info["optimizer_success"],
                                       "optimizer_status": info["optimizer_status"], "nfev": info["nfev"], "cost": info["cost"]})
                bound_rows.append({"model": model_name, "fold": fold, "parameter": name, "value": float(value),
                                   "lower_bound": float(lo), "upper_bound": float(hi),
                                   "at_lower_bound": bool(np.isclose(value, lo, rtol=0, atol=1e-6)),
                                   "at_upper_bound": bool(np.isclose(value, hi, rtol=0, atol=1e-6))})

    oof = pd.concat(all_predictions, ignore_index=True)
    metric_rows = []
    for model, group in oof.groupby("model", sort=False):
        valid = group.prediction_valid.to_numpy(bool)
        row = {
            "model": model,
            "n_galaxies": int(group.galaxy.nunique()),
            "n_rows": len(group),
            "n_valid_rows": int(valid.sum()),
            "n_invalid_rows": int((~valid).sum()),
            "invalid_row_fraction": float((~valid).mean()),
            "oof_prediction_domain_admissible": bool(valid.all()),
            **metrics(group.loc[valid, "log_gobs"].to_numpy(), group.loc[valid, "log_gpred"].to_numpy()),
        }
        metric_rows.append(row)
    metric_table = pd.DataFrame(metric_rows)

    galaxy_errors = (oof.groupby(["model", "galaxy"], as_index=False)
                     .agg(prediction_domain_admissible=("prediction_valid", "all"),
                          rmse_dex=("residual_dex", lambda x: float(np.sqrt(np.mean(np.square(x)))) if x.notna().all() else np.nan),
                          mae_dex=("residual_dex", lambda x: float(np.mean(np.abs(x))) if x.notna().all() else np.nan),
                          bias_dex=("residual_dex", lambda x: float(x.mean()) if x.notna().all() else np.nan),
                          n_rows=("residual_dex", "size"),
                          n_invalid_rows=("prediction_valid", lambda x: int((~x).sum()))))
    reference = galaxy_errors[galaxy_errors.model == REFERENCE_MODEL][["galaxy", "rmse_dex"]].rename(columns={"rmse_dex": "m19_rmse_dex"})
    rng = np.random.default_rng(SEED)
    ablation_rows = []
    for model in MODELS:
        candidate = galaxy_errors[galaxy_errors.model == model].merge(reference, on="galaxy", validate="one_to_one")
        delta = candidate.rmse_dex.to_numpy() - candidate.m19_rmse_dex.to_numpy()
        finite = np.isfinite(delta)
        delta_finite = delta[finite]
        if model == REFERENCE_MODEL:
            p_value = np.nan
        else:
            try:
                p_value = float(wilcoxon(delta_finite, zero_method="wilcox", alternative="two-sided", method="auto").pvalue)
            except ValueError:
                p_value = np.nan
        boot = (np.array([rng.choice(delta_finite, size=len(delta_finite), replace=True).mean() for _ in range(N_BOOTSTRAP)])
                if len(delta_finite) else np.array([np.nan]))
        ablation_rows.append({"model": model, "n_galaxies": len(delta), "n_complete_valid_galaxies": int(finite.sum()),
                              "n_domain_invalid_galaxies": int((~candidate.prediction_domain_admissible).sum()),
                              "mean_delta_rmse_dex_vs_m19": float(delta_finite.mean()) if len(delta_finite) else np.nan,
                              "median_delta_rmse_dex_vs_m19": float(np.median(delta_finite)) if len(delta_finite) else np.nan,
                              "bootstrap95_low": float(np.quantile(boot, 0.025)), "bootstrap95_high": float(np.quantile(boot, 0.975)),
                              "wilcoxon_two_sided_p": p_value,
                              "interpretation_rule": "positive delta means this complete comparison model has worse galaxy-wise OOF prediction than full M19; any domain-invalid OOF galaxy makes it non-admissible for a full comparison"})
    comparison = pd.DataFrame(ablation_rows)
    parameter_table, bound_table = pd.DataFrame(parameter_rows), pd.DataFrame(bound_rows)
    bound_summary = (bound_table.groupby(["model", "parameter"], as_index=False)
                     .agg(folds_at_lower_bound=("at_lower_bound", "sum"), folds_at_upper_bound=("at_upper_bound", "sum"), folds=("fold", "size")))
    parameter_summary = (parameter_table.groupby(["model", "parameter"], as_index=False)
                         .agg(fold_median=("value", "median"), fold_min=("value", "min"), fold_max=("value", "max"), fold_std=("value", "std")))

    # Plot only OOF differences; no training scores are used for interpretation.
    shown = comparison[comparison.model != REFERENCE_MODEL].copy()
    fig, ax = plt.subplots(figsize=(10, 5), dpi=170)
    yerr = np.vstack([shown.mean_delta_rmse_dex_vs_m19 - shown.bootstrap95_low, shown.bootstrap95_high - shown.mean_delta_rmse_dex_vs_m19])
    ax.errorbar(np.arange(len(shown)), shown.mean_delta_rmse_dex_vs_m19, yerr=yerr, fmt="o", color="black", capsize=3)
    ax.axhline(0, color="tab:red", lw=1)
    ax.set_xticks(np.arange(len(shown)), shown.model.str.replace("_", " "), rotation=30, ha="right")
    ax.set_ylabel("mean galaxy-wise ΔRMSE [dex] versus full M19")
    ax.set_title("M19 joint minimal-model comparison: galaxy-separated OOF validation")
    fig.tight_layout()
    fig.savefig(figures / "fig_stage6t45_joint_minimal_model_oof_delta_rmse.png", bbox_inches="tight")
    plt.close(fig)

    source_record = pd.DataFrame([{"source_path": str(source), "source_sha256": sha256(source), **{f"column_{key}": value for key, value in mapping.items()}}])
    folds = pd.DataFrame(sorted(fold_for_galaxy.items()), columns=["galaxy", "outer_fold"])
    source_record.to_csv(tables / "stage6t45_source_lock.csv", index=False)
    pd.DataFrame(source_candidates).to_csv(tables / "stage6t45_source_discovery_candidates.csv", index=False)
    folds.to_csv(tables / "stage6t45_locked_galaxy_folds.csv", index=False)
    oof.to_csv(tables / "stage6t45_oof_predictions_all_models.csv", index=False)
    metric_table.to_csv(tables / "stage6t45_oof_metrics.csv", index=False)
    galaxy_errors.to_csv(tables / "stage6t45_galaxywise_oof_errors.csv", index=False)
    comparison.to_csv(tables / "stage6t45_comparison_vs_full_m19.csv", index=False)
    parameter_table.to_csv(tables / "stage6t45_fold_parameters.csv", index=False)
    parameter_summary.to_csv(tables / "stage6t45_parameter_stability_summary.csv", index=False)
    bound_table.to_csv(tables / "stage6t45_parameter_bound_audit.csv", index=False)
    bound_summary.to_csv(tables / "stage6t45_parameter_bound_summary.csv", index=False)

    joint_row = comparison.loc[comparison.model == JOINT_MODEL].iloc[0].to_dict()
    joint_p = float(joint_row["wilcoxon_two_sided_p"])
    joint_mean_delta = float(joint_row["mean_delta_rmse_dex_vs_m19"])
    joint_admissible = int(joint_row["n_domain_invalid_galaxies"]) == 0
    joint_significant = bool(joint_admissible and np.isfinite(joint_p) and joint_p < 0.05)
    if joint_significant and joint_mean_delta < 0:
        joint_verdict = "The joint q=0.5 linear-gradient variant is significantly better than full M19 in this post-selection internal OOF comparison. It is a candidate compact empirical model, not a confirmed final equation."
    elif joint_significant and joint_mean_delta > 0:
        joint_verdict = "The joint q=0.5 linear-gradient variant is significantly worse than full M19 in this post-selection internal OOF comparison. The two small single changes do not combine into an improvement."
    elif joint_admissible:
        joint_verdict = "No statistically reliable total OOF difference between the joint q=0.5 linear-gradient variant and full M19 is established in this post-selection internal comparison."
    else:
        joint_verdict = "The joint q=0.5 linear-gradient variant has held-out domain failures and is not fully comparable with full M19."

    decision = [{"stage": STAGE, "code_version": CODE_VERSION, "analysis_type": "internal_galaxy_separated_out_of_fold_joint_minimal_model_comparison",
                 "external_unopened_target_claimed": False, "same_folds_for_all_models": True,
                 "test_fold_used_for_optimization": False, "oof_predictions_saved": True,
                 "models_tested": len(MODELS), "source_modified": False,
                 "galaxy_varying_stellar_ml_available": False,
                 "population_ml_component_tested": False,
                 "fixed_sparc_stellar_ml_already_embedded_in_gbar": True,
                 "comparison_was_post_selection_from_stage6t44b": True,
                 "joint_model": JOINT_MODEL,
                 "joint_model_domain_admissible": joint_admissible,
                 "joint_model_mean_delta_rmse_dex_vs_full_m19": joint_mean_delta,
                 "joint_model_wilcoxon_two_sided_p": joint_p,
                 "joint_model_significant_at_0_05": joint_significant,
                 "fundamental_uqsh_field_equation_claimed": False,
                 "verdict": joint_verdict,
                 "next_allowed_step": "Do not claim a final equation from this post-selection test. Freeze the resulting candidate only after a separately specified model-selection rule, then test it without refitting on an independent rotation-curve dataset. A population/M-L ablation separately requires a provenance-locked, galaxy-level M/L catalogue."}]
    pd.DataFrame(decision).to_csv(tables / "stage6t45_decision.csv", index=False)
    (root / "stage6t45_manifest.json").write_text(json.dumps({"stage": STAGE, "code_version": CODE_VERSION, "run_id": run_id, "source": str(source), "source_sha256": sha256(source), "models": MODELS, "reference_model": REFERENCE_MODEL, "joint_model": JOINT_MODEL, "outer_folds": OUTER_FOLDS, "seed": SEED, "n_starts": N_STARTS, "max_nfev": MAX_NFEV}, indent=2), encoding="utf-8")
    (reports / "stage6t45_report.txt").write_text(
        "STAGE 6T45 -- M19 JOINT MINIMAL-MODEL COMPARISON\n\n"
        "This is an internal SPARC galaxy-separated OOF comparison, not an external prospective validation.\n"
        "The q=0.5 plus no-saturation combination was selected after Stage 6T44B and is therefore exploratory/post-selection.\n"
        "The population/M-L component is not tested: the canonical radial source has no independently varying stellar M/L column.\n"
        "Positive mean delta RMSE (comparison model minus full M19) means the comparison model cost predictive performance.\n"
        "A non-positive or non-finite held-out prediction is recorded as a domain failure; it is not clipped and the affected model is not fully comparable.\n\n"
        + "JOINT VERDICT\n" + joint_verdict + "\n\n"
        + comparison.to_string(index=False) + "\n\nOOF METRICS\n" + metric_table.to_string(index=False) + "\n",
        encoding="utf-8",
    )
    print("=" * 100)
    print("STAGE 6T45 -- M19 JOINT MINIMAL-MODEL COMPARISON WITH GALAXY-GROUP-CV")
    print("=" * 100)
    print("CODE VERSION:", CODE_VERSION)
    print("SOURCE:", source)
    print("OOF METRICS")
    print(metric_table.to_string(index=False))
    print("\nCOMPARISON VS FULL M19")
    print(comparison.to_string(index=False))
    print("\nJOINT VERDICT")
    print(joint_verdict)
    print("\nOUTPUT:", root)


if __name__ == "__main__":
    main()

STAGE 6T45 -- M19 JOINT MINIMAL-MODEL COMPARISON WITH GALAXY-GROUP-CV
CODE VERSION: 6T45.1-joint-minimal-model-20260730
SOURCE: /content/drive/MyDrive/UQSH_DarkMatter_Project/data_processed/sparc_radial_observables.csv
OOF METRICS
                             model  n_galaxies  n_rows  n_valid_rows  n_invalid_rows  invalid_row_fraction  oof_prediction_domain_admissible  rmse_dex  mae_dex  median_abs_error_dex       r2  bias_dex
M19_full_free_q_saturated_gradient         175    3389          3389               0                   0.0                              True  0.194150 0.136442              0.100007 0.881502 -0.017514
   C1_fixed_q05_saturated_gradient         175    3389          3389               0                   0.0                              True  0.191858 0.133777              0.096734 0.884283 -0.015677
         C2_free_q_linear_gradient         175    3389          3389               0                   0.0                              True  0.192367 0.133509       

In [4]:
"""STAGE 6T47 -- EXACT ALGEBRA EQUIVALENCE AUDIT FOR COMPACT C3.

This is deliberately not another model-selection exercise.  In each
galaxy-separated fold it fits *only* compact C3,

    omega = g_bar**(1-eta) * g_cum**eta
    delta = K * sqrt(omega)
    g_pred = g_bar + delta + lambda d(r delta)/dr.

It then maps those exact fitted parameters into the old C3 notation,

    delta = A*g_ref*sqrt(omega/g_ref),

using A=1 and g_ref=K**2.  No second fit is performed.  Thus any difference
between the two predictions can only be floating-point arithmetic, not a
different optimum.  This audit establishes only algebraic equivalence; it
does not select C3 over full M19 or make an external-data claim.
"""
from __future__ import annotations

import hashlib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import least_squares
from sklearn.model_selection import GroupKFold

try:
    from google.colab import drive  # type: ignore
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass

STAGE = "6T47"
CODE_VERSION = "6T47.1-exact-algebra-equivalence-20260730"
PROJECT = Path("/content/drive/MyDrive/UQSH_DarkMatter_Project")
INPUT = PROJECT / "data_processed" / "sparc_radial_observables.csv"
AUDIT = PROJECT / "journal_audit"
OUTER_FOLDS = 5
SEED = 20260730
N_STARTS = 8
MAX_NFEV = 4000
# This is a numerical tolerance only.  The two expressions are evaluated
# independently, hence one or two floating-point roundings are unavoidable.
LOG_TOL_DEX = 1e-12
GPRED_REL_TOL = 1e-12

ALIASES = {
    "galaxy": ("galaxy", "galaxy_name", "name"),
    "radius": ("r_kpc", "radius_kpc", "radius", "r"),
    "gbar": ("gbar_m_s2", "g_bar_m_s2", "gbar"),
    "gobs": ("gobs_m_s2", "g_obs_m_s2", "gobs"),
}


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def norm(value: object) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def column(raw: pd.DataFrame, logical: str) -> str:
    available = {norm(c): str(c) for c in raw.columns}
    for name in ALIASES[logical]:
        if norm(name) in available:
            return available[norm(name)]
    raise ValueError(f"Missing {logical}; available columns: {list(raw.columns)}")


def cumulative_gbar(radius: np.ndarray, gbar: np.ndarray) -> np.ndarray:
    integral = np.zeros_like(radius, dtype=float)
    if len(radius) > 1:
        y = radius * gbar
        integral[1:] = np.cumsum(0.5 * (y[1:] + y[:-1]) * np.diff(radius))
    return np.maximum(2.0 * integral / np.maximum(radius**2, 1e-300), 1e-300)


def prepare() -> tuple[pd.DataFrame, dict[str, str]]:
    if not INPUT.is_file():
        raise FileNotFoundError(f"Canonical source not found: {INPUT}")
    raw = pd.read_csv(INPUT)
    mapping = {key: column(raw, key) for key in ALIASES}
    data = pd.DataFrame({
        "galaxy": raw[mapping["galaxy"]].astype(str).str.strip(),
        "radius": pd.to_numeric(raw[mapping["radius"]], errors="coerce"),
        "gbar": pd.to_numeric(raw[mapping["gbar"]], errors="coerce"),
        "gobs": pd.to_numeric(raw[mapping["gobs"]], errors="coerce"),
    }).replace([np.inf, -np.inf], np.nan).dropna()
    data = data[(data.radius > 0) & (data.gbar > 0) & (data.gobs > 0)].copy()
    data = data.sort_values(["galaxy", "radius"]).reset_index(drop=True)
    if (data.groupby("galaxy").radius.diff().fillna(1.0) <= 0).any():
        raise RuntimeError("Non-increasing radii make the radial derivative undefined.")
    data["gcum"] = data.groupby("galaxy", group_keys=False).apply(
        lambda x: pd.Series(cumulative_gbar(x.radius.to_numpy(float), x.gbar.to_numpy(float)), index=x.index),
        include_groups=False,
    ).sort_index()
    data["log_gobs"] = np.log10(data.gobs)
    return data, mapping


def omega(frame: pd.DataFrame, eta: float) -> np.ndarray:
    logged = ((1.0 - eta) * np.log(frame.gbar.to_numpy(float))
              + eta * np.log(frame.gcum.to_numpy(float)))
    return np.exp(np.clip(logged, -700.0, 700.0))


def gradient(frame: pd.DataFrame, source: np.ndarray) -> np.ndarray:
    result = np.full(len(frame), np.nan)
    radii = frame.radius.to_numpy(float)
    for pos in frame.groupby("galaxy", sort=False).indices.values():
        i = np.asarray(pos, dtype=int)
        result[i] = np.gradient(radii[i] * source[i], radii[i])
    return result


def compact_predict(frame: pd.DataFrame, values: np.ndarray) -> np.ndarray:
    log_k, eta, lam = values
    source = 10.0**log_k * np.sqrt(np.maximum(omega(frame, eta), 1e-300))
    return frame.gbar.to_numpy(float) + source + lam * gradient(frame, source)


def legacy_predict(frame: pd.DataFrame, values: np.ndarray) -> np.ndarray:
    log_a, log_gref, eta, lam = values
    gref = 10.0**log_gref
    source = 10.0**log_a * gref * np.sqrt(np.maximum(omega(frame, eta) / gref, 1e-300))
    return frame.gbar.to_numpy(float) + source + lam * gradient(frame, source)


def compact_to_legacy(values: np.ndarray) -> np.ndarray:
    """Use A=1 and g_ref=K^2, so A*sqrt(g_ref)=K exactly by definition."""
    log_k, eta, lam = values
    return np.array([0.0, 2.0 * log_k, eta, lam], dtype=float)


def fit_compact(train: pd.DataFrame, seed: int) -> tuple[np.ndarray, dict[str, object]]:
    low, high = np.array([-11., 0., -4.]), np.array([-1., 1., 4.])
    weight = 1.0 / train.groupby("galaxy").galaxy.transform("size").to_numpy(float)
    def residual(values: np.ndarray) -> np.ndarray:
        pred = compact_predict(train, values)
        if np.any(~np.isfinite(pred)) or np.any(pred <= 0):
            return np.full(len(train), 1e6)
        return (np.log10(pred) - train.log_gobs.to_numpy(float)) * np.sqrt(weight)
    rng = np.random.default_rng(seed)
    starts = [np.array([-5.25, 0.5, 0.0])] + [low + rng.random(3) * (high - low) for _ in range(N_STARTS - 1)]
    results = [least_squares(residual, x0, bounds=(low, high), method="trf", max_nfev=MAX_NFEV) for x0 in starts]
    best = min(results, key=lambda r: float(np.mean(residual(r.x) ** 2)))
    return best.x, {"success": bool(best.success), "nfev": int(best.nfev), "cost": float(best.cost)}


def self_test() -> None:
    probe = pd.DataFrame({"galaxy": ["A"] * 4, "radius": [1., 2., 3., 4.],
                          "gbar": [2e-11, 1.6e-11, 1.2e-11, 1e-11],
                          "gcum": [2e-11, 1.8e-11, 1.5e-11, 1.3e-11]})
    compact = np.array([-5.25, 0.37, -0.2])
    legacy = compact_to_legacy(compact)
    p1, p2 = compact_predict(probe, compact), legacy_predict(probe, legacy)
    if not np.allclose(p1, p2, rtol=GPRED_REL_TOL, atol=0.0):
        raise RuntimeError("Preflight failed: parameter mapping is not algebraically equivalent.")


def main() -> None:
    self_test()
    data, mapping = prepare()
    galaxies = np.array(sorted(data.galaxy.unique()))
    if len(galaxies) < 20:
        raise RuntimeError("Too few galaxies for the locked group-CV audit.")
    folds = {}
    for fold, (_, test_i) in enumerate(GroupKFold(n_splits=OUTER_FOLDS).split(galaxies, groups=galaxies)):
        folds.update({str(g): fold for g in galaxies[test_i]})
    data["outer_fold"] = data.galaxy.map(folds).astype(int)
    parts, rows = [], []
    for fold in range(OUTER_FOLDS):
        train, test = data[data.outer_fold != fold].reset_index(drop=True), data[data.outer_fold == fold].reset_index(drop=True)
        compact, info = fit_compact(train, SEED + fold)
        legacy = compact_to_legacy(compact)
        p_compact, p_legacy = compact_predict(test, compact), legacy_predict(test, legacy)
        if np.any(p_compact <= 0) or np.any(p_legacy <= 0):
            raise RuntimeError(f"Fold {fold}: compact C3 has an invalid prediction; equivalence is not evaluable.")
        part = test[["galaxy", "radius", "gbar", "gobs", "gcum", "outer_fold", "log_gobs"]].copy()
        part["gpred_compact"] = p_compact
        part["gpred_legacy_from_exact_mapping"] = p_legacy
        part["logpred_compact"] = np.log10(p_compact)
        part["logpred_legacy_from_exact_mapping"] = np.log10(p_legacy)
        part["abs_log_prediction_difference_dex"] = np.abs(part.logpred_compact - part.logpred_legacy_from_exact_mapping)
        part["relative_gpred_difference"] = np.abs(p_compact - p_legacy) / np.maximum(np.abs(p_compact), 1e-300)
        parts.append(part)
        rows.append({"fold": fold, "log10_K": compact[0], "eta": compact[1], "lambda": compact[2],
                     "mapped_log10_A": legacy[0], "mapped_log10_gref": legacy[1],
                     "mapping": "A=1; log10(gref)=2*log10(K)", **info,
                     "n_train_galaxies": train.galaxy.nunique(), "n_test_galaxies": test.galaxy.nunique()})
    oof = pd.concat(parts, ignore_index=True)
    per_galaxy = oof.groupby("galaxy", as_index=False).agg(
        max_abs_log_prediction_difference_dex=("abs_log_prediction_difference_dex", "max"),
        max_relative_gpred_difference=("relative_gpred_difference", "max"),
        n_rows=("galaxy", "size"))
    max_log = float(oof.abs_log_prediction_difference_dex.max())
    max_rel = float(oof.relative_gpred_difference.max())
    passed = bool(max_log <= LOG_TOL_DEX and max_rel <= GPRED_REL_TOL)
    run = "run_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_UTC")
    root = AUDIT / "uqsh_stage6t47_m19_exact_algebra_equivalence_audit" / run
    tables, reports = root / "tables", root / "reports"
    tables.mkdir(parents=True, exist_ok=False); reports.mkdir(parents=True, exist_ok=False)
    pd.DataFrame([{"source_path": str(INPUT), "source_sha256": sha256(INPUT), **{f"column_{k}": v for k, v in mapping.items()}}]).to_csv(tables / "stage6t47_source_lock.csv", index=False)
    pd.DataFrame(sorted(folds.items()), columns=["galaxy", "outer_fold"]).to_csv(tables / "stage6t47_locked_galaxy_folds.csv", index=False)
    pd.DataFrame(rows).to_csv(tables / "stage6t47_compact_fit_and_exact_mapping.csv", index=False)
    oof.to_csv(tables / "stage6t47_exact_mapped_prediction_comparison.csv", index=False)
    per_galaxy.to_csv(tables / "stage6t47_galaxywise_numerical_difference.csv", index=False)
    decision = pd.DataFrame([{
        "compact_form_fitted_once_per_fold": True, "legacy_form_refitted": False,
        "exact_mapping": "log10(A)=0; log10(gref)=2*log10(K); eta and lambda unchanged",
        "max_abs_log_prediction_difference_dex": max_log, "predeclared_log_tolerance_dex": LOG_TOL_DEX,
        "max_relative_gpred_difference": max_rel, "predeclared_relative_tolerance": GPRED_REL_TOL,
        "algebraic_equivalence_passes_numerical_audit": passed,
        "verdict": ("PASS: compact C3 and legacy C3 are algebraically equivalent under the exact parameter mapping."
                    if passed else "FAIL: inspect implementation or numerical stability before using the compact notation."),
        "scope": "Parameterization audit only; no external validation and no model-selection claim.",
    }])
    decision.to_csv(tables / "stage6t47_equivalence_decision.csv", index=False)
    (root / "stage6t47_manifest.json").write_text(json.dumps({"stage": STAGE, "code_version": CODE_VERSION, "source_sha256": sha256(INPUT), "outer_folds": OUTER_FOLDS, "seed": SEED, "log_tolerance_dex": LOG_TOL_DEX, "relative_tolerance": GPRED_REL_TOL}, indent=2), encoding="utf-8")
    (reports / "stage6t47_report.txt").write_text(decision.to_string(index=False) + "\n", encoding="utf-8")
    print("=" * 100); print("STAGE 6T47 -- EXACT ALGEBRA EQUIVALENCE AUDIT FOR COMPACT C3"); print("=" * 100)
    print("CODE VERSION:", CODE_VERSION); print("SOURCE:", INPUT); print("\nEXACT-MAPPING DECISION"); print(decision.to_string(index=False)); print("\nOUTPUT:", root)


if __name__ == "__main__":
    main()

STAGE 6T47 -- EXACT ALGEBRA EQUIVALENCE AUDIT FOR COMPACT C3
CODE VERSION: 6T47.1-exact-algebra-equivalence-20260730
SOURCE: /content/drive/MyDrive/UQSH_DarkMatter_Project/data_processed/sparc_radial_observables.csv

EXACT-MAPPING DECISION
 compact_form_fitted_once_per_fold  legacy_form_refitted                                                exact_mapping  max_abs_log_prediction_difference_dex  predeclared_log_tolerance_dex  max_relative_gpred_difference  predeclared_relative_tolerance  algebraic_equivalence_passes_numerical_audit                                                                                        verdict                                                                             scope
                              True                 False log10(A)=0; log10(gref)=2*log10(K); eta and lambda unchanged                           8.881784e-15                   1.000000e-12                   2.338185e-14                    1.000000e-12                                    